In [1]:
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)


In [2]:
import numpy as np
import matplotlib.pyplot as plt
from tensorly.decomposition import parafac
from numpy.polynomial.chebyshev import chebval
import tensorly as tl

# Set backend
tl.set_backend('numpy')

# Define 3D function
def f(xyz, c=5):
    return 1 / (1 + c**2 * np.sum(xyz**2, axis=1))

# Generate 3D Chebyshev polynomials
from numpy.polynomial.chebyshev import chebval

def chebyshev_polys(x, deg):
    coeffs = np.eye(deg + 1)
    return np.array([chebval(x, coeff) for coeff in coeffs])


# Step 1: Generate 3D coefficient tensor via least squares
def generate_coeff_tensor(N, c):
    k = np.arange(N+1)
    nodes = np.cos((2*k + 1) * np.pi / (2*(N+1)))
    X, Y, Z = np.meshgrid(nodes, nodes, nodes, indexing="ij")
    
    coords = np.stack([X.ravel(), Y.ravel(), Z.ravel()], axis=1)
    F = f(coords, c).reshape((N+1, N+1, N+1))

    Tx = chebyshev_polys(nodes, N)
    Ty = Tx.copy()
    Tz = Tx.copy()

    A = np.kron(Tz.T, np.kron(Ty.T, Tx.T))
    F_flat = F.ravel()
    c_flat, *_ = np.linalg.lstsq(A, F_flat, rcond=None)
    C = c_flat.reshape((N+1, N+1, N+1))
    return C, nodes

# Step 2: Evaluate function from CP-decomposed Chebyshev tensor
def evaluate_cp_interp(weights, factors, nodes, N, resolution=50):
    xx = np.linspace(-1, 1, resolution)
    yy = np.linspace(-1, 1, resolution)
    zz = np.linspace(-1, 1, resolution)
    Tx = chebyshev_polys(xx, N)
    Ty = chebyshev_polys(yy, N)
    Tz = chebyshev_polys(zz, N)
    F_cp = np.zeros((resolution, resolution, resolution))

    for r in range(len(weights)):
        a = Tx.T @ factors[0][:, r]
        b = Ty.T @ factors[1][:, r]
        c = Tz.T @ factors[2][:, r]
        F_cp += weights[r] * np.einsum("i,j,k->ijk", a, b, c)
    
    return F_cp

# Step 3: Compute ground truth on a dense grid
def compute_exact_function_grid(f, c, resolution=50):
    xx = np.linspace(-1, 1, resolution)
    yy = np.linspace(-1, 1, resolution)
    zz = np.linspace(-1, 1, resolution)
    X, Y, Z = np.meshgrid(xx, yy, zz, indexing="ij")
    coords = np.stack([X.ravel(), Y.ravel(), Z.ravel()], axis=1)
    F = f(coords, c).reshape((resolution, resolution, resolution))
    return F

In [3]:
# Parameters
N = 16
c = 5
resolution = 50
ranks = range(1, 11)

# Get 3D coefficient tensor
C, nodes = generate_coeff_tensor(N, c)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorly.decomposition import parafac
import tensorly as tl
from numpy.polynomial.chebyshev import chebval

# Set TensorLy backend
tl.set_backend('numpy')

# Define 3D function to interpolate
def f(xyz, c=5):
    return 1 / (1 + c**2 * np.sum(xyz**2, axis=1))

# Generate Chebyshev polynomials using numpy's chebval
def chebyshev_polys(x, deg):
    coeffs = np.eye(deg + 1)
    return np.array([chebval(x, coeff) for coeff in coeffs])  # shape: (deg+1, len(x))

# Memory-safe coefficient tensor generator using projection
def generate_coeff_tensor_projection(N, c):
    k = np.arange(N + 1)
    nodes = np.cos((2 * k + 1) * np.pi / (2 * (N + 1)))
    X, Y, Z = np.meshgrid(nodes, nodes, nodes, indexing="ij")

    coords = np.stack([X.ravel(), Y.ravel(), Z.ravel()], axis=1)
    F = f(coords, c).reshape((N + 1, N + 1, N + 1))

    Tx = chebyshev_polys(nodes, N)  # (N+1, N+1)
    Ty = Tx.copy()
    Tz = Tx.copy()

    # Project onto Chebyshev basis
    C = np.einsum("il,jm,kn,ijk->lmn", Tx, Ty, Tz, F)
    # C /= (N + 1)**3  # scale for Chebyshev grid

    return C, nodes

# Evaluate Chebyshev interpolant from CP-decomposed tensor
def evaluate_cp_interp(weights, factors, N, resolution=50):
    xx = np.linspace(-1, 1, resolution)
    yy = np.linspace(-1, 1, resolution)
    zz = np.linspace(-1, 1, resolution)
    Tx = chebyshev_polys(xx, N)
    Ty = chebyshev_polys(yy, N)
    Tz = chebyshev_polys(zz, N)

    F_cp = np.zeros((resolution, resolution, resolution))
    for r in range(len(weights)):
        a = Tx.T @ factors[0][:, r]
        b = Ty.T @ factors[1][:, r]
        c = Tz.T @ factors[2][:, r]
        F_cp += weights[r] * np.einsum("i,j,k->ijk", a, b, c)

    return F_cp

# Evaluate interpolant directly using original tensor (no CP)
def evaluate_direct_interp(C, N, resolution=50):
    xx = np.linspace(-1, 1, resolution)
    yy = np.linspace(-1, 1, resolution)
    zz = np.linspace(-1, 1, resolution)
    Tx = chebyshev_polys(xx, N)
    Ty = chebyshev_polys(yy, N)
    Tz = chebyshev_polys(zz, N)
    return np.einsum("ia,jb,kc,abc->ijk", Tx.T, Ty.T, Tz.T, C)

# Ground truth evaluator
def compute_exact_function_grid(f, c, resolution=50):
    xx = np.linspace(-1, 1, resolution)
    yy = np.linspace(-1, 1, resolution)
    zz = np.linspace(-1, 1, resolution)
    X, Y, Z = np.meshgrid(xx, yy, zz, indexing="ij")
    coords = np.stack([X.ravel(), Y.ravel(), Z.ravel()], axis=1)
    return f(coords, c).reshape((resolution, resolution, resolution))

# Main script
N = 64
c = 5
resolution = 50
ranks = range(1, 11)

# Generate coefficients
C, nodes = generate_coeff_tensor_projection(N, c)
F_true = compute_exact_function_grid(f, c, resolution)

# Baseline: direct interpolation (no CP)
F_direct = evaluate_direct_interp(C, N, resolution)
diff_direct = F_true - F_direct
rmse_baseline = np.sqrt(np.mean(diff_direct**2))
maxe_baseline = np.max(np.abs(diff_direct))
print(f"[Baseline] RMSE: {rmse_baseline:.2e}, MaxE: {maxe_baseline:.2e}")

# CP decomposition + error analysis
rmse_errors = []
maxe_errors = []

for rank in ranks:
    cp_tensor = parafac(C, rank=rank, init='svd')
    weights, factors = cp_tensor.weights, cp_tensor.factors

    F_cp = evaluate_cp_interp(weights, factors, N, resolution)
    diff = F_true - F_cp
    rmse = np.sqrt(np.mean(diff**2))
    maxe = np.max(np.abs(diff))

    rmse_errors.append(rmse)
    maxe_errors.append(maxe)
    print(f"Rank {rank}: RMSE = {rmse:.2e}, MaxE = {maxe:.2e}")

# Plot
fig, axs = plt.subplots(1, 2, figsize=(14, 5))

axs[0].plot(ranks, rmse_errors, marker='o')
axs[0].set_yscale("log")
axs[0].set_title("3D CP: RMSE vs Rank")
axs[0].set_xlabel("CP Rank")
axs[0].set_ylabel("RMSE")
axs[0].grid(True, which='both', linestyle='--')

axs[1].plot(ranks, maxe_errors, marker='s', color='red')
axs[1].set_yscale("log")
axs[1].set_title("3D CP: MaxE vs Rank")
axs[1].set_xlabel("CP Rank")
axs[1].set_ylabel("MaxE")
axs[1].grid(True, which='both', linestyle='--')

plt.tight_layout()
plt.show()


In [9]:

import numpy as np
import tensorly as tl
from tensorly.decomposition import parafac

tl.set_backend('numpy')


In [10]:
def f(xyz, c=5):
    return 1 / (1 + c**2 * np.sum(xyz**2, axis=1))

def chebyshev_polys(x, deg):
    T = np.zeros((deg+1, len(x)))
    T[0] = 1
    if deg > 0:
        T[1] = x
    for k in range(2, deg+1):
        T[k] = 2 * x * T[k-1] - T[k-2]
    return T


In [ ]:
def generate_coeff_tensor(N, c):
    k = np.arange(N+1)
    nodes = np.cos((2*k + 1) * np.pi / (2*(N+1)))
    X, Y, Z = np.meshgrid(nodes, nodes, nodes, indexing="ij")
    coords = np.stack([X.ravel(), Y.ravel(), Z.ravel()], axis=1)
    F = f(coords, c).reshape((N+1, N+1, N+1))

    T = chebyshev_polys(nodes, N)  # (N+1, N+1)

    # Raw projection (no normalization yet)
    C_raw = np.einsum('ia,jb,kc,ijk->abc', T, T, T, F)

    # Per-axis scaling vectors: s[0]=1/(N+1), s[n>=1]=2/(N+1)
    s = np.full(N+1, 2.0/(N+1))
    s[0] = 1.0/(N+1)

    # Apply along each mode (outer product of scalings)
    C = C_raw * s[:, None, None] * s[None, :, None] * s[None, None, :]

    return C, nodes

In [12]:
def evaluate_cp_interp(weights, factors, nodes, N, resolution=50):
    xx = np.linspace(-1, 1, resolution)
    yy = np.linspace(-1, 1, resolution)
    zz = np.linspace(-1, 1, resolution)
    Tx = chebyshev_polys(xx, N)
    Ty = chebyshev_polys(yy, N)
    Tz = chebyshev_polys(zz, N)
    F_cp = np.zeros((resolution, resolution, resolution))

    for r in range(len(weights)):
        a = Tx.T @ factors[0][:, r]
        b = Ty.T @ factors[1][:, r]
        c = Tz.T @ factors[2][:, r]
        F_cp += weights[r] * np.einsum("i,j,k->ijk", a, b, c)

    return F_cp

def compute_exact_function_grid(f, c, resolution=50):
    xx = np.linspace(-1, 1, resolution)
    yy = np.linspace(-1, 1, resolution)
    zz = np.linspace(-1, 1, resolution)
    X, Y, Z = np.meshgrid(xx, yy, zz, indexing="ij")
    coords = np.stack([X.ravel(), Y.ravel(), Z.ravel()], axis=1)
    F = f(coords, c).reshape((resolution, resolution, resolution))
    return F

def evaluate_direct_chebyshev_interp(C, nodes, N, resolution=50):
    xx = np.linspace(-1, 1, resolution)
    yy = np.linspace(-1, 1, resolution)
    zz = np.linspace(-1, 1, resolution)
    Tx = chebyshev_polys(xx, N)
    Ty = chebyshev_polys(yy, N)
    Tz = chebyshev_polys(zz, N)
    F_interp = np.einsum("ia,jb,kc,abc->ijk", Tx.T, Ty.T, Tz.T, C)
    return F_interp


In [13]:
N = 64
c = 5
resolution = 100
ranks = range(1, 11)

C, nodes = generate_coeff_tensor(N, c)
F_true = compute_exact_function_grid(f, c, resolution)


In [14]:
ranks = range(1, 21)


for rank in ranks:
    cp_tensor = parafac(C, rank=rank, init='svd')
    weights, factors = cp_tensor.weights, cp_tensor.factors

    F_cp = evaluate_cp_interp(weights, factors, nodes, N, resolution)

    diff_func = F_true - F_cp
    rmse_func = np.sqrt(np.mean(diff_func**2))
    maxe_func = np.max(np.abs(diff_func))

    C_reconstructed = tl.cp_to_tensor(cp_tensor)
    l2_norm_coeff = np.linalg.norm(C - C_reconstructed)

    print(f"Rank {rank}: RMSE = {rmse_func:.2e}, MaxE = {maxe_func:.2e}, L2 Norm = {l2_norm_coeff:.2e}")


Rank 1: RMSE = 4.35e+03, MaxE = 1.63e+04, L2 Norm = 1.55e+03
Rank 2: RMSE = 4.44e+03, MaxE = 2.77e+04, L2 Norm = 3.17e+02
Rank 3: RMSE = 4.44e+03, MaxE = 2.93e+04, L2 Norm = 2.52e+02
Rank 4: RMSE = 4.45e+03, MaxE = 3.04e+04, L2 Norm = 5.22e+01
Rank 5: RMSE = 4.45e+03, MaxE = 3.06e+04, L2 Norm = 4.94e+01
Rank 6: RMSE = 4.45e+03, MaxE = 3.08e+04, L2 Norm = 2.54e+01
Rank 7: RMSE = 4.45e+03, MaxE = 3.06e+04, L2 Norm = 4.31e+01
Rank 8: RMSE = 4.45e+03, MaxE = 3.07e+04, L2 Norm = 2.99e+01
Rank 9: RMSE = 4.45e+03, MaxE = 3.08e+04, L2 Norm = 1.64e+01
Rank 10: RMSE = 4.45e+03, MaxE = 3.08e+04, L2 Norm = 8.37e+00
Rank 11: RMSE = 4.45e+03, MaxE = 3.08e+04, L2 Norm = 7.90e+00
Rank 12: RMSE = 4.45e+03, MaxE = 3.08e+04, L2 Norm = 6.52e+00
Rank 13: RMSE = 4.45e+03, MaxE = 3.08e+04, L2 Norm = 4.14e+00
Rank 14: RMSE = 4.45e+03, MaxE = 3.09e+04, L2 Norm = 2.73e+00
Rank 15: RMSE = 4.45e+03, MaxE = 3.09e+04, L2 Norm = 1.95e+00
Rank 16: RMSE = 4.45e+03, MaxE = 3.09e+04, L2 Norm = 1.98e+00
Rank 17: RMSE = 4

In [ ]:
F_interp_baseline = evaluate_direct_chebyshev_interp(C, nodes, N, resolution)
diff_baseline = F_true - F_interp_baseline
rmse_baseline = np.sqrt(np.mean(diff_baseline**2))
maxe_baseline = np.max(np.abs(diff_baseline))

print(f"[Baseline] RMSE: {rmse_baseline:.2e}, MaxE: {maxe_baseline:.2e}")


In [8]:
import numpy as np
import matplotlib.pyplot as plt
from numpy.polynomial.chebyshev import chebvander
from autograd import grad
import autograd.numpy as anp
from mpl_toolkits.mplot3d import Axes3D

# Function to approximate
def f(x, y, c=5):
    return 1 / (1 + c**2 * (x**2 + y**2))

# Settings
batch_size = 3
num_batches = 1000
N_total = batch_size * num_batches  # maintain consistent update count
d_x, d_y = 63, 63
R = 10
lr = 1e-2
num_iters = 500

# Generate full dataset
np.random.seed(42)
x_all = np.random.uniform(-1, 1, size=N_total)
y_all = np.random.uniform(-1, 1, size=N_total)
f_all = f(x_all, y_all)

# Split: 80% train, 20% test
split_idx = int(0.8 * N_total)
x_train, x_test = x_all[:split_idx], x_all[split_idx:]
y_train, y_test = y_all[:split_idx], y_all[split_idx:]
f_train, f_test = f_all[:split_idx], f_all[split_idx:]

# Chebyshev basis
Tx_train = chebvander(x_train, d_x)
Ty_train = chebvander(y_train, d_y)
Tx_test = chebvander(x_test, d_x)
Ty_test = chebvander(y_test, d_y)
Tx_train_a = anp.array(Tx_train)
Ty_train_a = anp.array(Ty_train)
f_train_a = anp.array(f_train)

# Vectorize parameters
def pack_params(A, B, lambdas):
    return anp.concatenate([A.ravel(), B.ravel(), lambdas])

def unpack_params(params):
    A_size = (d_x + 1) * R
    B_size = (d_y + 1) * R
    A = params[:A_size].reshape((d_x + 1, R))
    B = params[A_size:A_size + B_size].reshape((d_y + 1, R))
    lambdas = params[-R:]
    return A, B, lambdas

# Loss function with batchwise squared error
def make_loss_fn(Tx, Ty, f_true, batch_size):
    Tx_a = anp.array(Tx)
    Ty_a = anp.array(Ty)
    f_true_a = anp.array(f_true)

    def loss(params):
        A, B, lambdas = unpack_params(params)
        A_eval = Tx_a @ A
        B_eval = Ty_a @ B
        preds = anp.sum(lambdas * A_eval * B_eval, axis=1)
        sq_errors = (f_true_a - preds)
        total_loss = 0.0
        for i in range(0, len(f_true), batch_size):
            batch_error = anp.sum(sq_errors[i:i + batch_size])
            total_loss += batch_error ** 2
        total_loss /= (len(f_true) // batch_size)
        return total_loss

    return loss

# Train with Adam
loss_fn = make_loss_fn(Tx_train, Ty_train, f_train, batch_size)
grad_loss = grad(loss_fn)

A_init = 0.01 * np.random.randn(d_x + 1, R)
B_init = 0.01 * np.random.randn(d_y + 1, R)
lambdas_init = np.ones(R)
params = pack_params(A_init, B_init, lambdas_init)

m = anp.zeros_like(params)
v = anp.zeros_like(params)

for t in range(1, num_iters + 1):
    grads = grad_loss(params)
    m = 0.9 * m + 0.1 * grads
    v = 0.999 * v + 0.001 * (grads ** 2)
    m_hat = m / (1 - 0.9 ** t)
    v_hat = v / (1 - 0.999 ** t)
    params -= lr * m_hat / (anp.sqrt(v_hat) + 1e-8)

A_opt, B_opt, lambdas_opt = unpack_params(params)

# Evaluation (train)
A_eval_train = Tx_train @ A_opt
B_eval_train = Ty_train @ B_opt
train_preds = np.sum(lambdas_opt * A_eval_train * B_eval_train, axis=1)
rmse_train = np.sqrt(np.mean((f_train - train_preds)**2))
maxe_train = np.max(np.abs(f_train - train_preds))

# Evaluation (test)
A_eval_test = Tx_test @ A_opt
B_eval_test = Ty_test @ B_opt
test_preds = np.sum(lambdas_opt * A_eval_test * B_eval_test, axis=1)
rmse_test = np.sqrt(np.mean((f_test - test_preds)**2))
maxe_test = np.max(np.abs(f_test - test_preds))

# Print errors
print(f"Train RMSE = {rmse_train:.4e}, MaxE = {maxe_train:.4e}")
print(f"Test  RMSE = {rmse_test:.4e}, MaxE = {maxe_test:.4e}")

# Plotting
fig = plt.figure(figsize=(14, 10))

# True Train
ax1 = fig.add_subplot(221, projection='3d')
ax1.plot_trisurf(x_train, y_train, f_train, cmap='viridis', linewidth=0.2)
ax1.set_title("True Train Function")

# Predicted Train
ax2 = fig.add_subplot(222, projection='3d')
ax2.plot_trisurf(x_train, y_train, train_preds, cmap='viridis', linewidth=0.2)
ax2.set_title("Predicted Train Function")

# True Test
ax3 = fig.add_subplot(223, projection='3d')
ax3.plot_trisurf(x_test, y_test, f_test, cmap='plasma', linewidth=0.2)
ax3.set_title("True Test Function")

# Predicted Test
ax4 = fig.add_subplot(224, projection='3d')
ax4.plot_trisurf(x_test, y_test, test_preds, cmap='plasma', linewidth=0.2)
ax4.set_title("Predicted Test Function")

plt.tight_layout()
plt.show()


ModuleNotFoundError: No module named 'matplotlib'